# Module 5: Vision product prototype

**Project:** build an image classifier for a product idea of your own. You collect the photos, train a model from scratch as a baseline, then fine-tune a **pretrained** model and show how much better it does with little data.

**Done when:**
- you have a dataset of **at least 3 classes with 50 or more images each**,
- your fine-tuned model scores **at least as well as the from-scratch model on the test set** (usually far better; within one test image counts as a tie) and clearly above chance,
- you produced a **confusion matrix** and reviewed the model's **worst mistakes**.

**How to use this notebook**
1. Turn on a GPU: *Runtime > Change runtime type > T4 GPU*, then save your own copy (*File > Save a copy in Drive*).
2. Collect your photos first (Step 1). Do not skip this: a vision product starts with its data.
3. Run cells from top to bottom. Fill in every `YOUR CODE HERE`. The **Self-check** cells tell you if you got it right.
4. Stuck? Use the hint ladder: docs, then the hints below each step, then the course forum. Only after your own attempt, open the solution page.

## Step 0: Setup (given)

In [ ]:
import os, random, time, hashlib, zipfile, shutil
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224        # the size ImageNet models expect
NUM_WORKERS = 2
print("Using device:", device)

def set_seed(seed=0):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

loss_fn = nn.CrossEntropyLoss()

## Step 1: Collect your dataset

Pick a small product idea where a camera could help, for example: sorting recyclables, recognizing plant diseases, spotting product defects, telling food types apart, or identifying parts. Choose **3 or more classes** that you can photograph yourself.

**Rules for a good dataset**
- **At least 50 images per class** (more is better).
- Vary the **background, lighting, angle and distance**. A model learns what you show it, including accidental shortcuts such as "all cat photos are on the sofa".
- **One object class per photo.** No duplicates (copies of the same photo end up in both train and test, as you learned in Module 4).
- Phone photos are fine. Large images are fine, they are resized automatically.

**Folder layout:** one folder per class, all inside a folder named `my_data`:
```
my_data/
  class_a/  photo1.jpg  photo2.jpg ...
  class_b/  ...
  class_c/  ...
```
Zip the `my_data` folder as `my_data.zip`, then upload it with the **Files** panel on the left of Colab (the folder icon, then the upload button).

**No photos yet?** Set `USE_DEMO = True` in the next cell to build a small demo dataset from CIFAR-10 (tiny 32 pixel images, so the results will be rougher). Use your own photos for the real project.

In [ ]:
DATA_DIR = Path("my_data")
USE_DEMO = False     # True: build a demo dataset instead of using your own photos

def make_demo_dataset(root="my_data", classes=("airplane", "frog", "truck"), per_class=80):
    cifar = datasets.CIFAR10("cifar_raw", train=True, download=True)
    root = Path(root); counts = {c: 0 for c in classes}
    for img, label in cifar:
        name = cifar.classes[label]
        if name in counts and counts[name] < per_class:
            (root / name).mkdir(parents=True, exist_ok=True)
            img.save(root / name / f"{name}_{counts[name]:03d}.png")
            counts[name] += 1
        if all(v >= per_class for v in counts.values()):
            break

if USE_DEMO:
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    make_demo_dataset(DATA_DIR)
elif not DATA_DIR.exists() and Path("my_data.zip").exists():
    with zipfile.ZipFile("my_data.zip") as z:
        z.extractall(".")

def dataset_report(root):
    """Count images per class and look for unreadable files and exact duplicates."""
    root = Path(root)
    classes = sorted(d.name for d in root.iterdir() if d.is_dir() and not d.name.startswith("."))
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    counts, bad, seen, dups = {}, [], {}, []
    for c in classes:
        files = [f for f in (root / c).rglob("*") if f.suffix.lower() in exts]
        counts[c] = len(files)
        for f in files:
            try:
                with Image.open(f) as im:
                    im.verify()
            except Exception:
                bad.append(str(f)); continue
            h = hashlib.md5(f.read_bytes()).hexdigest()
            if h in seen: dups.append((str(f), seen[h]))
            else: seen[h] = str(f)
    return {"classes": classes, "counts": counts, "bad": bad, "dups": dups}

assert DATA_DIR.exists(), "Folder my_data not found. Upload my_data.zip (or set USE_DEMO = True) and run this cell again."
report = dataset_report(DATA_DIR)
for c in report["classes"]:
    print(f"{c:20s} {report['counts'][c]:4d} images")
print("unreadable files:", len(report["bad"]), "  exact duplicates:", len(report["dups"]))

### Self-check: dataset

In [ ]:
assert len(report["classes"]) >= 3, f"Found {len(report['classes'])} classes. You need at least 3."
_small = {c: n for c, n in report["counts"].items() if n < 50}
assert not _small, f"These classes have fewer than 50 images: {_small}. Collect more."
assert not report["bad"], f"Unreadable image files: {report['bad'][:3]}. Remove or replace them."
assert not report["dups"], f"{len(report['dups'])} duplicate images found, for example {report['dups'][0]}. Delete the copies, because duplicates leak between train and test."
print(f"Dataset check passed: {len(report['classes'])} classes, {sum(report['counts'].values())} images.")

## Step 2: Split the data

Write `stratified_split(labels, val_frac, test_frac, seed)`. It returns three lists of indices: `(train_idx, val_idx, test_idx)`.

**Stratified** means every class is split in the same proportions, so the test set is not accidentally missing a class. For each class: shuffle its indices, give `test_frac` of them to test, `val_frac` to validation, and the rest to training. (Use `round()` for the counts.)

*Hint:* `np.unique(labels)` lists the classes; `np.where(labels == c)[0]` gives the indices of class `c`; `rng.shuffle(idx)` shuffles in place.

In [ ]:
def stratified_split(labels, val_frac=0.15, test_frac=0.15, seed=0):
    """Return (train_idx, val_idx, test_idx), each a list of indices. Every class is split in the same proportions."""
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    train_idx, val_idx, test_idx = [], [], []
    # YOUR CODE HERE
    return train_idx, val_idx, test_idx

### Self-check: stratified_split

In [ ]:
_labels = np.array([0] * 100 + [1] * 60 + [2] * 40)
_tr, _va, _te = stratified_split(_labels, 0.15, 0.15, seed=0)
assert len(_tr) + len(_va) + len(_te) == 200, "Every example must be in exactly one split."
assert not (set(_tr) & set(_va)) and not (set(_tr) & set(_te)) and not (set(_va) & set(_te)), "The splits overlap. An example may be in only one split."
assert sorted(_tr + _va + _te) == list(range(200)), "Some examples are missing from all splits."
for _c, _n in ((0, 100), (1, 60), (2, 40)):
    for _name, _s, _frac in (("test", _te, 0.15), ("validation", _va, 0.15)):
        _got = int(np.sum(_labels[_s] == _c))
        assert abs(_got - _n * _frac) <= 1, f"Class {_c} has {_got} examples in {_name} but about {_n * _frac:.0f} were expected. Split each class separately."
assert stratified_split(_labels, seed=0) == stratified_split(_labels, seed=0), "The same seed should give the same split."
assert stratified_split(_labels, seed=0) != stratified_split(_labels, seed=1), "A different seed should give a different split."
print("stratified_split() check passed.")

## Step 3: Transforms and data augmentation

A pretrained model expects images in a specific format: the right size, and pixel values **normalized with the ImageNet mean and standard deviation**.

Write two transform pipelines with `transforms.Compose`:

- **`train_tfms`** (with augmentation): `RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0))`, `RandomHorizontalFlip()`, a mild `ColorJitter(0.2, 0.2, 0.2)`, then `ToTensor()` and `Normalize(imagenet_mean, imagenet_std)`.
- **`eval_tfms`** (no randomness): `Resize(int(IMG_SIZE * 1.14))`, `CenterCrop(IMG_SIZE)`, then `ToTensor()` and `Normalize(...)`.

**Why two?** Augmentation creates new variety for training. Validation and test images must be left alone, so the scores are repeatable and honest.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_tfms = None   # YOUR CODE HERE
eval_tfms = None    # YOUR CODE HERE

### Self-check: transforms

In [ ]:
assert train_tfms is not None and eval_tfms is not None, "Define both train_tfms and eval_tfms."
_img = Image.fromarray((np.random.default_rng(0).random((120, 160, 3)) * 255).astype("uint8"))
_a, _b = train_tfms(_img), train_tfms(_img)
_c, _d = eval_tfms(_img), eval_tfms(_img)
assert _a.shape == (3, IMG_SIZE, IMG_SIZE) and _c.shape == (3, IMG_SIZE, IMG_SIZE), f"Output should have shape (3, {IMG_SIZE}, {IMG_SIZE}). Check the crop and ToTensor."
assert not torch.equal(_a, _b), "train_tfms should be random (augmentation): two calls on the same image must differ."
assert torch.equal(_c, _d), "eval_tfms must NOT be random: two calls on the same image must be identical."
_black = Image.new("RGB", (160, 120), (0, 0, 0))     # a black pixel becomes about -2 after ImageNet normalization
assert eval_tfms(_black).min() < -1.5, "eval_tfms did not normalize pixel values with the ImageNet mean and std. Did you add Normalize(...) after ToTensor()?"
assert train_tfms(_black).min() < -1.5, "train_tfms did not normalize pixel values with the ImageNet mean and std. Add Normalize(...) after ToTensor() here too."
print("transforms check passed.")

### Build the datasets and loaders (given)

One `ImageFolder` is created with the training transforms and another with the evaluation transforms. The split decides which images go where, so **only the training images get augmented**.

In [ ]:
base_train = datasets.ImageFolder(DATA_DIR, transform=train_tfms)
base_eval = datasets.ImageFolder(DATA_DIR, transform=eval_tfms)
class_names = base_eval.classes
K = len(class_names)

tr_idx, va_idx, te_idx = stratified_split(base_eval.targets, 0.15, 0.15, seed=0)
train_ds, val_ds, test_ds = Subset(base_train, tr_idx), Subset(base_eval, va_idx), Subset(base_eval, te_idx)

def make_loaders(batch_size=32):
    return (DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS),
            DataLoader(val_ds, batch_size=64, num_workers=NUM_WORKERS),
            DataLoader(test_ds, batch_size=64, num_workers=NUM_WORKERS))

print(f"classes: {class_names}")
print(f"train {len(train_ds)}   validation {len(val_ds)}   test {len(test_ds)}")
print(f"chance accuracy for {K} classes: {1 / K:.2f}")

## Step 4: A baseline trained from scratch (given)

First see what a model can do with **only your small dataset and no pretraining**. This uses the `fit()` loop from Module 3, included here so the notebook stands alone. (Paste your own version instead if you prefer.)

In [ ]:
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            total_loss += loss_fn(logits, y).item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += y.size(0)
    return total_loss / n, correct / n

def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item() * y.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        n += y.size(0)
    return total_loss / n, correct / n

def fit(model, train_loader, val_loader, loss_fn, optimizer, epochs, device,
        patience=None, ckpt_path="best.pt"):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss, best_epoch, bad = float("inf"), -1, 0
    for epoch in range(epochs):
        tr = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        va = evaluate(model, val_loader, loss_fn, device)
        for k, v in zip(history, [tr[0], tr[1], va[0], va[1]]):
            history[k].append(v)
        print(f"epoch {epoch + 1:2d}: train acc {tr[1]:.3f}  val acc {va[1]:.3f}  val loss {va[0]:.3f}")
        if va[0] < best_val_loss:
            best_val_loss, best_epoch, bad = va[0], epoch, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad += 1
            if patience is not None and bad >= patience:
                break
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    history["best_epoch"], history["best_val_loss"] = best_epoch, best_val_loss
    return history

In [ ]:
set_seed(0)
train_dl, val_dl, test_dl = make_loaders(32)
scratch = models.resnet18(weights=None, num_classes=K).to(device)      # random weights: no pretraining
opt = torch.optim.AdamW(scratch.parameters(), lr=1e-3, weight_decay=1e-2)
t0 = time.time()
hist_scratch = fit(scratch, train_dl, val_dl, loss_fn, opt, epochs=12, device=device, patience=4, ckpt_path="scratch.pt")
scratch_time = time.time() - t0
scratch_test_acc = evaluate(scratch, test_dl, loss_fn, device)[1]
print(f"\nFROM SCRATCH: test accuracy {scratch_test_acc:.3f}   (chance {1 / K:.2f})   training time {scratch_time:.0f}s")

## Step 5: Transfer learning: a pretrained model with a new head

A model trained on ImageNet (1.2 million images) has already learned edges, textures and shapes. We keep that knowledge (the **backbone**) and replace only the final layer (the **head**) so it predicts *your* classes.

Write `build_finetune_model(num_classes, freeze_backbone)`:
1. Start from the pretrained ResNet-18 (already loaded for you in the first line).
2. If `freeze_backbone` is True, set `requires_grad = False` for **every** parameter, so they are not updated. Do this **before** step 3.
3. Replace `model.fc` with a new `nn.Linear(model.fc.in_features, num_classes)`. A newly created layer is trainable by default.

*Hint:* the order matters. Freeze everything first, then add the new head, so the head is the only part that trains.

In [ ]:
def build_finetune_model(num_classes, freeze_backbone=True):
    """ResNet-18 with pretrained ImageNet weights and a new classification head."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    # YOUR CODE HERE
    return model.to(device)

### Self-check: build_finetune_model

In [ ]:
_m = build_finetune_model(K, freeze_backbone=True)
_out = _m(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device))
assert _out.shape == (2, K), f"The model should output {K} scores per image, but gave shape {tuple(_out.shape)}. Replace model.fc with a Linear layer that has {K} outputs."
_trainable = [n for n, p in _m.named_parameters() if p.requires_grad]
assert set(_trainable) == {"fc.weight", "fc.bias"}, f"With the backbone frozen, ONLY the new head should be trainable, but these are trainable: {_trainable[:4]}"
_m2 = build_finetune_model(K, freeze_backbone=False)
assert all(p.requires_grad for p in _m2.parameters()), "With freeze_backbone=False every parameter should be trainable."
_total = sum(p.numel() for p in _m.parameters()); _head = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"build_finetune_model() check passed. Frozen: {_head:,} of {_total:,} parameters will train ({100 * _head / _total:.2f}%).")
del _m, _m2

## Step 6: Two phases: train the head, then fine-tune everything

A common recipe for small datasets:
1. **Feature extraction:** keep the backbone frozen and train only the new head with a normal learning rate.
2. **Fine-tuning:** unfreeze everything and train a little more with a **much lower learning rate**, so the pretrained features are gently adjusted rather than destroyed.

First write `unfreeze(model)`, which makes every parameter trainable again.

In [ ]:
def unfreeze(model):
    """Make every parameter of the model trainable."""
    # YOUR CODE HERE

### Self-check: unfreeze

In [ ]:
_m = build_finetune_model(K, freeze_backbone=True)
assert not all(p.requires_grad for p in _m.parameters()), "Test setup problem: the backbone should start frozen."
unfreeze(_m)
assert all(p.requires_grad for p in _m.parameters()), "After unfreeze(), every parameter should have requires_grad = True."
print("unfreeze() check passed.")
del _m

Now run both phases. This cell is given.

In [ ]:
set_seed(0)
train_dl, val_dl, test_dl = make_loaders(32)
ft = build_finetune_model(K, freeze_backbone=True)

print("Phase 1: train only the new head")
opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, ft.parameters()), lr=1e-3, weight_decay=1e-2)
t0 = time.time()
hist_head = fit(ft, train_dl, val_dl, loss_fn, opt, epochs=6, device=device, ckpt_path="head.pt")
head_test_acc = evaluate(ft, test_dl, loss_fn, device)[1]

print("\nPhase 2: unfreeze and fine-tune everything with a low learning rate")
unfreeze(ft)
opt = torch.optim.AdamW(ft.parameters(), lr=1e-4, weight_decay=1e-2)
hist_full = fit(ft, train_dl, val_dl, loss_fn, opt, epochs=8, device=device, patience=3, ckpt_path="full.pt")
ft_time = time.time() - t0
ft_test_acc = evaluate(ft, test_dl, loss_fn, device)[1]
print(f"\nHEAD ONLY: test accuracy {head_test_acc:.3f}")
print(f"FINE-TUNED: test accuracy {ft_test_acc:.3f}   training time {ft_time:.0f}s")

### Self-check: fine-tuning

In [ ]:
assert np.isfinite(hist_full["best_val_loss"]), "The validation loss is nan. Is the learning rate for fine-tuning low enough (1e-4)?"
assert ft_test_acc > 1 / K + 0.25, f"Fine-tuned test accuracy {ft_test_acc:.3f} is barely above chance ({1 / K:.2f}). Check the normalization in your transforms and that the head was replaced."
print(f"Fine-tuning check passed. Test accuracy {ft_test_acc:.3f}.")

## Step 7: The confusion matrix and the worst mistakes

Accuracy is a single number. A **confusion matrix** shows *which* classes get mistaken for which. Write `confusion_matrix(model, loader, num_classes)`. It returns a `(K, K)` array where entry `[i, j]` counts the examples whose **true** class is `i` and **predicted** class is `j`. The diagonal holds the correct predictions.

*Hint:* loop over the loader in evaluation mode with `torch.no_grad()`, take `pred = model(x).argmax(1)`, and add 1 to `cm[true, pred]` for each example.

In [ ]:
def confusion_matrix(model, loader, num_classes):
    """Return a (K, K) array: entry [i, j] = number of examples of true class i predicted as class j."""
    model.eval()
    cm = np.zeros((num_classes, num_classes), dtype=int)
    # YOUR CODE HERE
    return cm

### Self-check: confusion_matrix

In [ ]:
cm = confusion_matrix(ft, test_dl, K)
assert cm.shape == (K, K), f"The confusion matrix should have shape ({K}, {K}), but has shape {cm.shape}."
assert cm.sum() == len(test_ds), f"The matrix counts {cm.sum()} examples but the test set has {len(test_ds)}. Count every example once."
_acc = evaluate(ft, test_dl, loss_fn, device)[1]
assert abs(np.trace(cm) / cm.sum() - _acc) < 1e-9, "The diagonal of the matrix should give the same accuracy as evaluate(). Rows are TRUE classes, columns are PREDICTED classes."
print("confusion_matrix() check passed.")

Now look at the results. This cell is given.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(K)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(K)); ax.set_yticklabels(class_names)
ax.set_xlabel("predicted class"); ax.set_ylabel("true class"); ax.set_title("Confusion matrix (test set)")
for i in range(K):
    for j in range(K):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.show()
for i, name in enumerate(class_names):
    print(f"{name:20s} recall {cm[i, i] / max(cm[i].sum(), 1):.2f}")

In [ ]:
def show_worst_mistakes(model, dataset, k=10):
    """Show the k wrong predictions the model was MOST confident about."""
    model.eval(); wrong = []
    mean = torch.tensor(imagenet_mean).view(3, 1, 1); std = torch.tensor(imagenet_std).view(3, 1, 1)
    with torch.no_grad():
        for i in range(len(dataset)):
            x, y = dataset[i]
            probs = torch.softmax(model(x.unsqueeze(0).to(device)), dim=1)[0].cpu()
            p = int(probs.argmax())
            if p != y:
                wrong.append((float(probs[p]), x * std + mean, y, p))
    wrong.sort(key=lambda t: -t[0])
    if not wrong:
        print("No mistakes on this set. Try a harder look: use the validation set or collect harder photos."); return
    wrong = wrong[:k]
    fig, axes = plt.subplots(1, len(wrong), figsize=(2.4 * len(wrong), 3), squeeze=False)
    for ax, (conf, img, y, p) in zip(axes[0], wrong):
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1)); ax.axis("off")
        ax.set_title(f"true: {class_names[y]}\npred: {class_names[p]}\nconf {conf:.2f}", fontsize=8)
    plt.show()

show_worst_mistakes(ft, test_ds, k=10)

**Look at your worst mistakes and write down 3 patterns.** Is the photo blurry? Is the object hidden? Are two classes genuinely similar? Is the background misleading? Is a label wrong? What data would you collect next to fix the most common mistake?

*Your analysis:*

## Step 8: Compare the three models

Put the results side by side. This cell is given. Then write your conclusion below it.

**A warning about small test sets:** with about 25 test images, one mistake changes the accuracy by about 4 points. So a difference of one or two images is just noise. The check below counts a result **within one test image** of the baseline as a tie. The more photos you collect, the more trustworthy your comparison becomes.

In [ ]:
trainable_head = sum(p.numel() for p in build_finetune_model(K, True).parameters() if p.requires_grad)
trainable_all = sum(p.numel() for p in ft.parameters())
print(f"{'model':26s} {'test acc':>9s} {'trainable params':>18s}")
print(f"{'from scratch':26s} {scratch_test_acc:9.3f} {trainable_all:18,d}")
print(f"{'pretrained, head only':26s} {head_test_acc:9.3f} {trainable_head:18,d}")
print(f"{'pretrained, fine-tuned':26s} {ft_test_acc:9.3f} {trainable_all:18,d}")
print(f"\ntraining time: scratch {scratch_time:.0f}s, fine-tuned {ft_time:.0f}s")

In [ ]:
assert ft_test_acc >= scratch_test_acc - 1.0 / len(test_ds) - 1e-9, (f"Fine-tuned ({ft_test_acc:.3f}) is clearly below the from-scratch model ({scratch_test_acc:.3f}). "
    "Check that pretrained weights were loaded in build_finetune_model(), and that the fine-tuning learning rate is low.")
print(f"Done! Pretrained fine-tuning scored {ft_test_acc:.3f} against {scratch_test_acc:.3f} from scratch, "
      f"a gain of {100 * (ft_test_acc - scratch_test_acc):.1f} points.")

**Conclusion (write your answer here):** How much did pretraining help with your small dataset? Why? What would you collect or change next to improve the product?

*Your conclusion:*

## Stretch challenge (optional)

Compare **two backbones** on accuracy versus speed. Replace ResNet-18 with a smaller or larger model, for example `models.mobilenet_v3_small(weights="DEFAULT")` or `models.efficientnet_b0(weights="DEFAULT")`. Each has a different final layer name (look at `print(model)`), so replace the right one. Measure two things for each: the test accuracy, and the **images per second** at prediction time (time a few batches with `torch.no_grad()`). Which would you ship on a phone? Write down what you found:

*Your notes:*

## Reflection and submission

Write three lines in your portfolio repo's `module-05/README.md`:
1. What product idea you built a classifier for, and your classes
2. What surprised you (about the data, or about how much pretraining helped)
3. What you would collect or change next

Add a screenshot of your confusion matrix. Then download this notebook (*File > Download > .ipynb*), push it to `module-05` in your portfolio repo (do **not** push the photos if they are private), and submit the link in Moodle.